In [ ]:
import pandas as pd
from collections import Counter

In [ ]:
df = pd.read_csv("../data/paragraph_turns_full.csv")
df = df[['sentence', 'collectiveAction', 'racialJustice', 'sentence_length']]
df.head(1)

In [ ]:
filenames = {
    "relief": "../data/annotation/processing_log_relief.txt",
    "pride": "../data/annotation/processing_log_pride.txt",
    "guilt": "../data/annotation/processing_log_guilt.txt",
    "excitement": "../data/annotation/processing_log_excitement.txt",
    "disappointment": "../data/annotation/processing_log_dissapointment.txt",  
}

label_counts = {}

for emotion, filepath in filenames.items():
    with open(filepath, "r") as f:
        labels = []
        for line in f:
            if "label=" in line:
                label = int(line.strip().split("label=")[-1])
                labels.append(label)
        label_counts[emotion] = Counter(labels)

for emotion, counts in label_counts.items():
    print(f"{emotion.capitalize()} label counts: {dict(counts)}")

In [ ]:
def extract_labels(filepath):
    labels = {}
    with open(filepath, "r") as f:
        for line in f:
            if "Row" in line and "label=" in line:
                parts = line.strip().split("label=")
                row_num = int(parts[0].split()[1].replace(":", ""))
                label = int(parts[1])
                labels[row_num] = label
    return labels

In [ ]:
emotion_labels = {emotion: extract_labels(path) for emotion, path in filenames.items()}

In [ ]:
for emotion, labels_dict in emotion_labels.items():
    df[emotion] = df.index.map(lambda i: labels_dict.get(i, 0))  

In [ ]:
df_filtered = df[df['sentence_length'] <= 10]
df_filtered = df_filtered[df_filtered['sentence_length'] > 2]
#df_collective_action = df_filtered[df_filtered['collectiveAction'] == 1]

In [ ]:
all_emotions = ['relief', 'pride', 'guilt', 'excitement', 'disappointment']
subset_df = df_filtered[df_filtered[all_emotions].any(axis=1)]
subset_df

In [ ]:
# def sample_balanced_rows(df, emotion, n_per_type=5, seed=0, exclude_indices=set()):
#     pred_1 = df[(df[emotion] == 1) & (~df.index.isin(exclude_indices))]
#     pred_0 = df[(df[emotion] == 0) & (~df.index.isin(exclude_indices))]

#     # Sample from each label category (proxy for TP, FN, FP, TN)
#     tp_like = pred_1.sample(n=n_per_type, random_state=seed)
#     fn_like = pred_0.sample(n=n_per_type, random_state=seed + 1)
#     fp_like = pred_1.sample(n=n_per_type, random_state=seed + 2)
#     tn_like = pred_0.sample(n=n_per_type, random_state=seed + 3)

#     combined = pd.concat([tp_like, fn_like, fp_like, tn_like])
#     combined = combined.drop_duplicates()
#     combined["target_emotion"] = emotion
#     return combined

In [ ]:
def sample_balanced_rows(df, emotion, seed=0, exclude_indices=set()):
    df = df[~df.index.isin(exclude_indices)]

    sampled_rows = []

    for ca_value in [0, 1]:
        subset = df[df['collectiveAction'] == ca_value]

        pred_1 = subset[subset[emotion] == 1]
        pred_0 = subset[subset[emotion] == 0]

        # For this CA value, take 5 from each predicted label
        pred_1_sample = pred_1.sample(n=5, random_state=seed + ca_value * 10 + 1)
        pred_0_sample = pred_0.sample(n=5, random_state=seed + ca_value * 10 + 2)

        sampled = pd.concat([pred_1_sample, pred_0_sample])
        sampled_rows.append(sampled)

    combined = pd.concat(sampled_rows).drop_duplicates()
    combined["target_emotion"] = emotion
    return combined

In [ ]:
all_emotions = ['relief', 'pride', 'guilt', 'excitement', 'disappointment']
final_sample = pd.DataFrame()
used_indices = set()

for i, emotion in enumerate(all_emotions):
    sample = sample_balanced_rows(
        df_filtered,
        emotion,
        seed=200 + i,
        exclude_indices=used_indices
    )
    used_indices.update(sample.index)
    final_sample = pd.concat([final_sample, sample])

In [ ]:
final_sample = final_sample.sample(frac=1, random_state=999).reset_index(drop=True)
final_sample[["sentence", "collectiveAction", "target_emotion"] + all_emotions].head(100)

In [ ]:
final_sample[["sentence", "target_emotion"] + all_emotions].to_csv("../data/emotion_annotation_full.csv", index=False)

## Evaluation

In [3]:
import pandas as pd
import krippendorff

theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")

emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

alpha_results = {}

# For each emotion, create a matrix of annotations and compute alpha
for col in emotion_columns:
    # Each row is one item; each column is one annotator
    data = [theodora[col].tolist(), arianna[col].tolist()]
    
    # Handle missing data if any
    data = krippendorff.alpha(reliability_data=data, level_of_measurement='nominal')
    
    alpha_results[col] = data

# Display results
for emotion, alpha in alpha_results.items():
    print(f"{emotion}: Krippendorff's alpha = {alpha:.3f}")

relief: Krippendorff's alpha = 0.754
pride: Krippendorff's alpha = 0.778
guilt: Krippendorff's alpha = 0.741
excitement: Krippendorff's alpha = 0.558
disappointment: Krippendorff's alpha = 0.700


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score

# Load gold and individual annotator files
gold = pd.read_csv("../data/emotion_annotation_full.csv")
theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")

# Emotion columns to evaluate
emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

# Store accuracy results
accuracy = {
    "Theodora": {},
    "Arianna": {}
}

# Loop through emotion columns and compute accuracy
for col in emotion_columns:
    accuracy["Theodora"][col] = accuracy_score(gold[col], theodora[col])
    accuracy["Arianna"][col] = accuracy_score(gold[col], arianna[col])

# Print results
print("Accuracy Compared to Gold Labels:\n")
for annotator, scores in accuracy.items():
    print(f"{annotator}:")
    for emotion, acc in scores.items():
        print(f"  {emotion}: {acc:.3f}")
    print()

Accuracy Compared to Gold Labels:

Theodora:
  relief: 0.840
  pride: 0.780
  guilt: 0.870
  excitement: 0.860
  disappointment: 0.790

Arianna:
  relief: 0.810
  pride: 0.710
  guilt: 0.870
  excitement: 0.840
  disappointment: 0.690



In [5]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix
)

# Load gold and individual annotator files
gold = pd.read_csv("../data/emotion_annotation_full.csv")
theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")

# Emotion columns to evaluate
emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

# Store results
results = {
    "Theodora": {},
    "Arianna": {}
}

# Function to compute all metrics
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn
    }

# Loop through emotion columns and compute metrics
for col in emotion_columns:
    results["Theodora"][col] = compute_metrics(gold[col], theodora[col])
    results["Arianna"][col] = compute_metrics(gold[col], arianna[col])

# Print results
for annotator, emotions in results.items():
    print(f"{annotator}:\n")
    for emotion, metrics in emotions.items():
        print(f"{emotion}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.3f}" if isinstance(value, float) else f"  {metric}: {value}")
        print()

Theodora:

relief:
  accuracy: 0.840
  precision: 0.500
  recall: 0.188
  f1: 0.273
  tp: 3
  fp: 3
  fn: 13
  tn: 81

pride:
  accuracy: 0.780
  precision: 0.529
  recall: 0.391
  f1: 0.450
  tp: 9
  fp: 8
  fn: 14
  tn: 69

guilt:
  accuracy: 0.870
  precision: 0.250
  recall: 0.091
  f1: 0.133
  tp: 1
  fp: 3
  fn: 10
  tn: 86

excitement:
  accuracy: 0.860
  precision: 0.500
  recall: 0.357
  f1: 0.417
  tp: 5
  fp: 5
  fn: 9
  tn: 81

disappointment:
  accuracy: 0.790
  precision: 0.550
  recall: 0.478
  f1: 0.512
  tp: 11
  fp: 9
  fn: 12
  tn: 68

Arianna:

relief:
  accuracy: 0.810
  precision: 0.286
  recall: 0.125
  f1: 0.174
  tp: 2
  fp: 5
  fn: 14
  tn: 79

pride:
  accuracy: 0.710
  precision: 0.364
  recall: 0.348
  f1: 0.356
  tp: 8
  fp: 14
  fn: 15
  tn: 63

guilt:
  accuracy: 0.870
  precision: 0.250
  recall: 0.091
  f1: 0.133
  tp: 1
  fp: 3
  fn: 10
  tn: 86

excitement:
  accuracy: 0.840
  precision: 0.400
  recall: 0.286
  f1: 0.333
  tp: 4
  fp: 6
  fn: 10
  tn

In [6]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support
)

# Load gold and individual annotator files
gold = pd.read_csv("../data/emotion_annotation_full.csv")
theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")

# Emotion columns to evaluate
emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

# Function to compute metrics
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return acc, precision, recall, f1

# Collect rows for the summary table
rows = []

for col in emotion_columns:
    acc, prec, rec, f1 = compute_metrics(gold[col], theodora[col])
    rows.append(["Theodora", col, acc, prec, rec, f1])
    
    acc, prec, rec, f1 = compute_metrics(gold[col], arianna[col])
    rows.append(["Arianna", col, acc, prec, rec, f1])

# Create the summary table
df_results = pd.DataFrame(rows, columns=[
    "Annotator", "Emotion", "Accuracy", "Precision", "Recall", "F1"
])

# Round for display
print(df_results.round(3))

  Annotator         Emotion  Accuracy  Precision  Recall     F1
0  Theodora          relief      0.84      0.500   0.188  0.273
1   Arianna          relief      0.81      0.286   0.125  0.174
2  Theodora           pride      0.78      0.529   0.391  0.450
3   Arianna           pride      0.71      0.364   0.348  0.356
4  Theodora           guilt      0.87      0.250   0.091  0.133
5   Arianna           guilt      0.87      0.250   0.091  0.133
6  Theodora      excitement      0.86      0.500   0.357  0.417
7   Arianna      excitement      0.84      0.400   0.286  0.333
8  Theodora  disappointment      0.79      0.550   0.478  0.512
9   Arianna  disappointment      0.69      0.318   0.304  0.311
